# CareerOps-4B clean QLoRA
T4×2 + torchrun. Dataset: `careerops-clean-train` + private `careerops-runtime-auth`.
Auth: private `hf_token` file (secrets optional; API push wipes secret attachments).

In [ ]:
!pip -q install -U "transformers>=4.51" peft trl bitsandbytes accelerate datasets huggingface_hub

In [ ]:
import shutil, glob
from pathlib import Path
import torch

cands = sorted(Path('/kaggle/input').glob('**/train_ddp.py'))
assert cands, 'train_ddp.py missing from dataset'
shutil.copy(cands[0], '/kaggle/working/train_ddp.py')
print('train script', cands[0])

auth = sorted(Path('/kaggle/input').glob('**/hf_token'))
print('hf_token files', len(auth))
assert auth, 'private careerops-runtime-auth dataset with hf_token missing'

n = torch.cuda.device_count()
names = [torch.cuda.get_device_name(i) for i in range(n)]
print('gpus', n, names)
assert n >= 2, f'Need T4x2, got {n} {names}'
assert all('T4' in x or 'Tesla T4' in x for x in names), names

train = glob.glob('/kaggle/input/**/careerops_train.jsonl', recursive=True)
print('train jsonl', train)
assert train

In [ ]:
!torchrun --nproc_per_node=2 /kaggle/working/train_ddp.py